# phase_3 (M3 C-KAN) on Kaggle — Drive-resumable

Trains the per-region concept→disease model on **precomputed BioViL-T features**
(M1 is frozen, so features are cached once and just loaded here). Implements
`docs/VERA_phase_3_4_5_spec.md`: bbox-masked attention pool → neck 512→128 →
69 concepts → 14 CheXpert, with a global-head gate and macro-F1 as the headline metric.

Code layout (mirrors phase_2): `src/` = importable libs (config, constants, model, …),
`scripts/` = numbered run-order entries (`4-train.py`, `5-eval.py`, …) that self-insert `src/`.

**Storage:** `/kaggle/working` is wiped between sessions unless you Commit, and a
session can die. We push the run dir to **Google Drive (rclone)** each epoch (and
every `SYNC_EVERY` steps); next session set `RESUME=1` to pull `last.pt` and continue.

**Inputs you must upload as Kaggle datasets:**
1. `phase_3/` code folder (bundles the two small label JSONs under `src/` — self-contained).
2. **m3_labels** — `data/m3_labels/*.npy` + `manifest.jsonl` (from `scripts/1-labels.py`, ~640 MB).
3. **features** — the BioViL-T cache `<image_id>.pt` or `.npy` (produced **externally by the M1
   collaborator**; format = `features.py` contract `[197,C]`, ~45 GB).

Settings → Accelerator **GPU**, Internet **On** (for rclone), then Run All.

## CONFIG — edit these

In [ ]:
import os
PHASE3_SRC = "/kaggle/working/repo/phase_3"       # cloned from GitHub (next cell), not a dataset
LABELS     = "/kaggle/input/m3-labels"            # folder with region_concepts.npy ... manifest.jsonl
FEATURES   = "/kaggle/input/mimic-biovilt-feats"  # folder of <image_id>.npy

MODE     = "A"        # spec 3.3:  A Direct (safe fallback) | B pure CBM | C Hybrid. Run A->B->C.
EPOCHS   = 40
RUN_NAME = "m3_A"     # match MODE: m3_A / m3_B / m3_C
USE_GLOBAL = 0        # 1 = also add BioViL-T global embedding as a 30th region (extra; off by default)
RESUME   = 0          # 1 from the 2nd session on

# ===== durable checkpoints (Google Drive via YOUR OAuth token — see next section) =====
DRIVE_FOLDER_ID = "1c6AJ3fjsL449kiMK4xiXfnzwzA4gIo0O"   # CHEX-DATA folder id
RCLONE_REMOTE   = "dhint:m3_runs"                       # = CHEX-DATA/m3_runs
SYNC_EVERY      = 0     # 0 = push each epoch; >0 = also push every N steps

for k, v in dict(PHASE3_SRC=PHASE3_SRC, LABELS=LABELS, FEATURES=FEATURES, MODE=MODE,
                 EPOCHS=EPOCHS, RUN_NAME=RUN_NAME, USE_GLOBAL=USE_GLOBAL, RESUME=RESUME,
                 DRIVE_FOLDER_ID=DRIVE_FOLDER_ID, RCLONE_REMOTE=RCLONE_REMOTE, SYNC_EVERY=SYNC_EVERY).items():
    os.environ[k] = str(v)
print("config set | mode", MODE, "| resume", RESUME)

In [ ]:
# --- get the code from GitHub (instead of uploading a code dataset). Internet ON. ---
!rm -rf /kaggle/working/repo && git clone -q https://github.com/hiennguyendang/phase_2_3_4_5 /kaggle/working/repo
!ls /kaggle/working/repo/phase_3 | head

## 1 — install rclone + auth via YOUR Google account (OAuth token)

Service accounts have **no Drive storage quota** and 403 on upload to a *My Drive* folder — so we
authenticate as **you** with an OAuth token (files upload to your own quota).

**One-time setup, on a machine WITH a browser:**
1. Install rclone (https://rclone.org/downloads/ or `winget install Rclone.Rclone`).
2. `rclone authorize "drive"` → log in with the account that **owns CHEX-DATA** → *Allow*.
3. Copy the whole `{...}` token JSON it prints.
4. Kaggle **Add-ons → Secrets → Add**: label **`GDRIVE_TOKEN`**, value = that `{...}`.
5. Keep the CHEX-DATA folder id in `DRIVE_FOLDER_ID` (CONFIG). The cell below runs a real write-test.

In [ ]:
%%bash
set -e
if ! command -v rclone >/dev/null 2>&1; then
  cd /kaggle/working && curl -sLO https://downloads.rclone.org/rclone-current-linux-amd64.zip
  unzip -q -o rclone-current-linux-amd64.zip && cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone
fi
rclone version | head -1

In [ ]:
import os
# Drive remote 'dhint' via YOUR OAuth token (NOT a service account: SA has no storage quota and
# 403s on upload to My Drive). Token = `rclone authorize "drive"` -> Kaggle Secret GDRIVE_TOKEN.
# Graceful: if missing/unwritable, training still runs WITHOUT sync.
SYNC_OK = "0"
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    token = sec.get_secret("GDRIVE_TOKEN").strip()
    os.environ["RCLONE_CONFIG_DHINT_TYPE"] = "drive"
    os.environ["RCLONE_CONFIG_DHINT_TOKEN"] = token
    os.environ["RCLONE_CONFIG_DHINT_SCOPE"] = "drive"
    os.environ["RCLONE_CONFIG_DHINT_ROOT_FOLDER_ID"] = os.environ["DRIVE_FOLDER_ID"]
    os.environ.pop("RCLONE_CONFIG_DHINT_SERVICE_ACCOUNT_FILE", None)  # drop stale SA from a prior run
    # OPTIONAL own Google API client -> private query quota (avoids shared-client RATE_LIMIT on long runs)
    for k_sec, k_env in [("GDRIVE_CLIENT_ID", "RCLONE_CONFIG_DHINT_CLIENT_ID"),
                         ("GDRIVE_CLIENT_SECRET", "RCLONE_CONFIG_DHINT_CLIENT_SECRET")]:
        try:
            os.environ[k_env] = sec.get_secret(k_sec).strip()
        except Exception:
            pass
    using_own = "own client" if os.environ.get("RCLONE_CONFIG_DHINT_CLIENT_ID") else "rclone shared client"
    remote = os.environ["RCLONE_REMOTE"]
    if os.system('rclone mkdir "%s"' % remote) == 0 and \
       os.system('echo ok | rclone rcat "%s/_write_test.txt"' % remote) == 0:
        SYNC_OK = "1"
        os.system('rclone delete "%s/_write_test.txt" 2>/dev/null' % remote)
        print(f"remote OK (OAuth, write verified, {using_own}) ->", remote)
    else:
        print("[WARN] Drive write FAILED -> check GDRIVE_TOKEN (own account, write scope) + DRIVE_FOLDER_ID")
except Exception as e:
    print("[WARN] GDRIVE_TOKEN secret not set -> NO Drive sync:", e)
os.environ["SYNC_OK"] = SYNC_OK
print("SYNC_OK =", SYNC_OK)

## 2 — copy code in

In [ ]:
!rm -rf /kaggle/working/phase_3 && cp -r "$PHASE3_SRC" /kaggle/working/phase_3
%cd /kaggle/working/phase_3
# code now lives in src/ (libs) + scripts/ (numbered run-order entries). Scripts self-insert src/
# on sys.path, so run them as scripts/N-*.py from here. Sanity-check the libs load:
!python -c "import sys; sys.path.insert(0,'src'); import constants as C; print('regions',C.NUM_REGIONS,'concepts',C.NUM_CONCEPTS)"

## 3 — train (Drive-synced)

Re-run with `RESUME=1` in CONFIG after a session dies — it pulls `last.pt` from Drive
and continues. Run modes **A, B, C** (change `MODE`/`RUN_NAME`) to compare. Checkpoint
selection is on **val image macro-F1** (spec 3.6); AUC is reported alongside.

In [ ]:
import os
G = "--use-global" if int(os.environ["USE_GLOBAL"]) else ""
R = "--resume" if int(os.environ["RESUME"]) else ""
S = ('--sync-remote "%s" --sync-every %s' % (os.environ["RCLONE_REMOTE"], os.environ["SYNC_EVERY"]))     if os.environ.get("SYNC_OK") == "1" else ""
os.environ.update(G_FLAG=G, R_FLAG=R, S_FLAG=S)
print("flags:", G, R, S or "(no sync)")
!python scripts/4-train.py --mode "$MODE" --labels-dir "$LABELS" --features-root "$FEATURES" \
  --out /kaggle/working/m3_runs --name "$RUN_NAME" --epochs $EPOCHS --device cuda \
  --workers 4 $G_FLAG $S_FLAG $R_FLAG

## 4 — eval (test = gold) + infer (per-region JSON for M4/M5)

In [ ]:
!python scripts/5-eval.py --ckpt /kaggle/working/m3_runs/$RUN_NAME/best.pt \
  --labels-dir "$LABELS" --features-root "$FEATURES" --split test --device cuda

In [ ]:
!python scripts/7-infer.py --ckpt /kaggle/working/m3_runs/$RUN_NAME/best.pt \
  --labels-dir "$LABELS" --features-root "$FEATURES" --split test \
  --out /kaggle/working/m3_pred_$RUN_NAME.jsonl --topk-cells 3 --device cuda
import os
if os.environ.get("SYNC_OK")=="1":
    os.system('rclone copy /kaggle/working/m3_runs/%s "%s/%s" --quiet' % (os.environ["RUN_NAME"], os.environ["RCLONE_REMOTE"], os.environ["RUN_NAME"]))
    print("final run pushed to Drive")

## 5 — faithfulness (spec 3.4) — decides which mode may claim "why"

Run **after** training B and/or C. The *faithfulness numbers* (not accuracy) decide:
- **B**: concept-intervention test — forcing a concept on/off must move its disease.
- **C**: leakage test — if zeroing the concept channel barely drops F1, concepts are decorative.
- Either: go/no-go concept-from-image F1; too low → ship **A** (where-faithful), demote concepts.

Set `RUN_NAME` to the run you want to probe (`m3_B` or `m3_C`).

In [ ]:
!python scripts/6-faithfulness.py --ckpt /kaggle/working/m3_runs/$RUN_NAME/best.pt \
  --labels-dir "$LABELS" --features-root "$FEATURES" --split val --device cuda